In [0]:
import requests
import re
import zipfile
import os
import io
import urllib.request 
from pyspark.sql.functions import current_timestamp
from pyspark.sql.functions import col, date_format

In [0]:
dbutils.widgets.text(
    "bronze_catalog",
     "dbr_dev")
dbutils.widgets.text(
    "bronze_schema",
    "artemzharkov10_bronze"
)

dbutils.widgets.text(
    "gold_catalog",
     "dbr_dev")
dbutils.widgets.text(
    "gold_schema",
    "artemzharkov10_gold"
)

BRONZE_CATALOG = dbutils.widgets.get("bronze_catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")

GOLD_CATALOG = dbutils.widgets.get("gold_catalog")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")



In [0]:
# dbutils.widgets.remove("catalog")

In [0]:


try: 
    MASTER_URL = dbutils.secrets.get(scope="default2", key="artem-library-link")
except Exception as e:
    print(f"Could not get secret: {e}")
# http://data.gdeltproject.org/gdeltv2/masterfilelist.txt (Link to library)
# Size(MB) /             MD5-hash                /      URL Link to download  /2015-02-18/minut/type          
# 150383 297a16b493de7cf6ca809a7cc31d0b93 http://data.gdeltproject.org/gdeltv2/20150218230000.export.CSV.zip

In [0]:
expected_gold_tables = [
    "finance_bitcoin_anomaly_gold"
]

#Just unique date
dates_to_download = set()

for table in expected_gold_tables:
    TABLE_PATH = f"{GOLD_CATALOG}.{GOLD_SCHEMA}.{table}"
    try:
        df_gold = spark.read.table(TABLE_PATH)
        df_date = df_gold.select(date_format(col("AnomalyDate"),"yyyyMMdd").alias("DateStr"))
        found_dates = []
        rows = df_date.collect()
        for row in rows:
            # print(row.DateStr)
            found_dates.append(row.DateStr)
        dates_to_download.update(found_dates)
    
    except Exception as e:
        print(f"Could not get table {TABLE_PATH}: {e}")
        # One target day for testing 2015-03-03

print(dates_to_download)


In [0]:

# limits files per one day
TARGET_URL = []
START_HOUR = 12
END_HOUR = 16

response = requests.get(MASTER_URL)
print("response status:", response.status_code)

if response.status_code == 200 and dates_to_download is not None:
    lines = response.text.splitlines()
    # print("Amount of lines: " + str(len(lines)))
    for line in lines:
        if "export.CSV.zip" in line:
            url = line.split(" ")[2]
            filename = url.split('/')[-1]  #20150218230000.export.CSV.zip
            file_date = filename[:8] # 20150218
            file_hour = int(filename[8:10]) # hours 23
            if file_date in dates_to_download:
                if START_HOUR < file_hour < END_HOUR:
                        TARGET_URL.append(url)

  



In [0]:
# for i in TARGET_URL:
#     print(i)
print(len(TARGET_URL))

In [0]:

VOLUME_PATH = f"/Volumes/{BRONZE_CATALOG}/{BRONZE_SCHEMA}/raw_data/gdelt_history"

try:  
    os.makedirs(VOLUME_PATH, exist_ok=True)
except Exception as e:
    print(f"Could not create directory {VOLUME_PATH}: {e}")

for url in TARGET_URL:
    zip_filename = url.split('/')[-1] # 20150218230000.export.CSV.zip
    csv_filename = zip_filename.replace('.zip', '') # 20150218230000.export.CSV
    target_file_path = os.path.join(VOLUME_PATH, csv_filename) #create link to Azure
    # Indenpondence
    if os.path.exists(target_file_path):
        print(f" Data: {csv_filename} is already exist in Volume")
        continue #skip
     #=============================================
    response = requests.get(url)    
    try:
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            # write in Azure  wb - write binary 
            with open(target_file_path, "wb") as f:
                file_to_write = z.open(csv_filename).read()
                f.write(file_to_write) 
    except Exception as e:
        print(f"Could not download file {url}: {e}")

In [0]:

df_raw = spark.read.option("delimiter", "\t").csv(VOLUME_PATH)
print(f"Count of raws in all downloaded files {df_raw.count()}")
display(df_raw.limit(10))